In [1]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft
#from sliced_wasserstein import sliced_wasserstein_distance
#from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import pickle

/home/marco/anaconda3/envs/MA_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [2]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import gc

import itertools
from scipy.stats import ttest_ind

from captum.attr import LayerGradientXActivation, LayerIntegratedGradients

from captum.concept import TCAV
from captum.concept import Concept

from captum.concept._utils.data_iterator import dataset_to_dataloader, CustomIterableDataset
from captum.concept._utils.common import concepts_to_str

from torch.utils.data import DataLoader, Dataset, IterableDataset, TensorDataset
from scipy.stats import gaussian_kde

input_shape_st = (60, 900)

CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))



         


In [3]:
def heteroscedastic_gaussian_nll(y_pred, y_true, log_vars):
    vars = torch.exp(log_vars)
    one_over_vars = 1 / (vars + 1e-4)
    nll = (log_vars + torch.clamp(one_over_vars * ((y_true - y_pred) ** 2), min=1e-4, max=1e3)).mean()
    return nll

def weighted_mse_loss(y_pred, y_true, kde_weights):
    return torch.mean(kde_weights * (y_pred - y_true) ** 2)

def compute_kde_weights(y_true, lambda2=0.5):
    y_true_np = y_true.cpu().numpy()
    
    if y_true_np.ndim == 0 or len(y_true_np) == 1:
        # If we have a single data point, return a weight of 1
        return torch.ones_like(y_true)
    
    kde = gaussian_kde(y_true_np)
    densities = kde(y_true_np)
    weights = 1 / (densities ** lambda2)
    
    # Clip weights at 95th percentile
    percentile_95 = np.percentile(weights, 95)
    weights = np.clip(weights, a_min=None, a_max=percentile_95)
    
    weights = weights / weights.sum() * len(weights)
    return torch.tensor(weights, device=y_true.device)

def total_loss_fn(y_pred, y_true, log_vars, beta=0.001, lambda2=0.5):
    kde_weights = compute_kde_weights(y_true, lambda2)
    
    # Weighted MSE loss
    mse_loss = weighted_mse_loss(y_pred, y_true, kde_weights)
    
    # Weighted heteroscedastic loss
    nll = beta * heteroscedastic_gaussian_nll(y_pred, y_true, log_vars)
    weighted_nll = (kde_weights * nll).mean()
    
    heteroscedastic_loss = beta * weighted_nll

    return heteroscedastic_loss + mse_loss

In [4]:
def total_loss_fn(y_pred, y_true, log_vars, beta=0.001, lambda2=0.5):
    kde_weights = compute_kde_weights(y_true, lambda2)
    
    # Weighted MSE loss
    mse_loss = weighted_mse_loss(y_pred, y_true, kde_weights)
    
    # Weighted heteroscedastic loss
    nll = beta * heteroscedastic_gaussian_nll(y_pred, y_true, log_vars)
    weighted_nll = (kde_weights * nll).mean()
    
    heteroscedastic_loss = beta * weighted_nll

    return heteroscedastic_loss + mse_loss

In [5]:
def load_pretrained_predictions(subject_index):
    # Load the predictions from the pretrained model
    predictions, uncertainties = np.load(f"pretrain_predictions_subject_{subject_index}.npy")
    return predictions, uncertainties

def load_pretrained_model():
    trunk_net = TrunkNet(n_chans=60, n_times=900)
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    model.load_state_dict(torch.load("/home/marco/Documents/GitHub/tms_eeg_decoding/data/checkpoint_full_epoch_1000_pen.pt").get('model_state_dict'))
    model.eval()
    model.to(device)
    return model


In [6]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model


In [7]:
def load_final_predictions(subject_index):
    # Load the predictions from the final model
    predictions, uncertainties = np.load(f"final_predictions_subject_{subject_index}.npy")
    return predictions, uncertainties


In [8]:
def load_continuous_predictions(subject_index):
    # Load the predictions from the continuous model
    with open(f'subject_{subject_index}_results.pkl', 'rb') as f:
        data = pickle.load(f)
    return data["predictions"]


In [9]:
def load_data_set(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index =  subject_index
    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names= load_eeg_data(cfg)
    return all_epochs[150:,:,:900], all_labels_raw[150:], ch_names, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep

In [10]:
def get_model_start_indices():
    # return a dictionary with subject index as key and start indices as values
    start_indices = {}
    cfg = load_config()
    for subject_index in cfg.dataset.test_subject_indices:
        all_epochs, _, _, _, _, _, _, _ = load_data_set(subject_index=subject_index)
        #indices = np.arange(100, all_epochs.shape[0]-51, 50)
        indices = np.array([], dtype=int)
        indices = np.append(indices, int(all_epochs.shape[0]+99))
        start_indices[subject_index] = indices

    return start_indices

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = load_config()
input_shape_st = (60, 900)

In [12]:
def evaluate_performance_final_pretrain(model, subj_index):
    cfg = load_config()
    results = []
    
    all_epochs, all_labels_raw, ch_names, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep =  load_data_set(subj_index)
    with torch.no_grad():
        data_loader = create_dataloader(all_epochs, (all_labels_raw >= fixed_median).astype(int), all_labels_raw, fixed_median, fixed_q1, fixed_q3, cfg.training.batch_size, mode='train')
    
        total_loss = 0
        mse_loss = 0
        mse_loss_total = 0
        correct_binary = 0
    
        for data in data_loader:
            data = {k: v.to(device) for k, v in data.items()}
            true_label = data['label_raw']

            if fixed_median is None:
                fixed_median = data['fixed_median'][0].item()
            outputs = model(data['epoch'])
            pred_label = outputs[:, 0]
            log_vars = outputs[:, 1]

            loss = total_loss_fn(pred_label, data['label_raw'], log_vars, beta=cfg.training.nll_beta, lambda2=cfg.training.kde_lambda)
            total_loss += loss.item()
            mse_loss = (pred_label - data['label_raw']) ** 2
            mse_loss_total += sum(mse_loss)
        

            # Binary classification using fixed median
            true_binary_fixed = true_label > fixed_median
            pred_binary_fixed = pred_label > fixed_median
            binary_correct_fixed = sum(true_binary_fixed == pred_binary_fixed)
            correct_binary += binary_correct_fixed

        results = {
            'total_loss': total_loss,
            'mse_loss': mse_loss_total,
            'binary_accuracy': (correct_binary / len(data_loader.dataset)),
     }
    
    return results

In [13]:
def evaluate_performance_continuous(subj_index):
    cfg = load_config()
    results = []
    
    all_epochs, all_labels_raw, ch_names, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep =  load_data_set(subj_index)
    cfg = load_config()
    results = []

    total_loss = 0
    correct_binary = 0
    mse_loss_total = 0
    with torch.no_grad():               
        for i in tqdm(range(0, len(all_epochs))):
            start_index = i+100
            inputs = torch.from_numpy(all_epochs[i])
            inputs = inputs.to(device).float()
            inputs = inputs.unsqueeze(0)
            true_label = torch.from_numpy(np.array(all_labels_raw[i])).to(device).float()
        # construct function to load correct model for current trial
            model = load_model(cfg,start_index=start_index, subject_index=subj_index)
            pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
            loss = total_loss_fn(pred_mean,  true_label, log_var, beta=cfg.training.nll_beta, lambda2=cfg.training.kde_lambda)
            total_loss += loss.item()
            mse_loss = (pred_mean - true_label) ** 2
            mse_loss_total += mse_loss

            # Binary classification using fixed median
            true_binary_fixed = all_labels_raw[i] > fixed_median
            pred_binary_fixed = pred_mean > fixed_median
            binary_correct_fixed = sum(true_binary_fixed == pred_binary_fixed)
            correct_binary += binary_correct_fixed

        results = {
            'total_loss': total_loss,
            'binary_accuracy': (correct_binary / len(all_epochs)),
            'mse_loss': mse_loss_total
        }
    return results

In [14]:
predictions_pretrain, uncertainties_pretrain = load_pretrained_predictions(2)
predictions_final, uncertainties_final = load_final_predictions(2)
predictions_continuous = load_continuous_predictions(2)

In [15]:
start_indices = get_model_start_indices()

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_013_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


...

Adding metadata with 1 columns
608 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_092_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_092_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
564 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [16]:
start_indices

{1: array([459]),
 2: array([670]),
 13: array([582]),
 24: array([484]),
 26: array([552]),
 27: array([474]),
 29: array([680]),
 34: array([733]),
 35: array([472]),
 41: array([713]),
 42: array([737]),
 43: array([676]),
 45: array([596]),
 46: array([654]),
 47: array([700]),
 48: array([449]),
 52: array([603]),
 55: array([606]),
 56: array([534]),
 57: array([621]),
 60: array([701]),
 62: array([722]),
 67: array([595]),
 69: array([548]),
 72: array([651]),
 73: array([569]),
 79: array([722]),
 80: array([709]),
 86: array([685]),
 88: array([557]),
 92: array([513]),
 102: array([738])}

In [17]:
pretrained_model = load_pretrained_model()
final_model = load_model(cfg, start_index=start_indices[2][0], subject_index=2)

/tmp/ipykernel_691660/460464609.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home/marco/Documents/GitHub/tms_eeg_decoding/data/che

In [18]:
evaluate_performance_final_pretrain(pretrained_model, 2)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...


    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


{'total_loss': 41.07218806094558,
 'mse_loss': tensor cuda:0 1.838e+03,
 'binary_accuracy': tensor cuda:0 0.525}

In [19]:
evaluate_performance_final_pretrain(final_model, 2)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


{'total_loss': 0.55651222637612,
 'mse_loss': tensor cuda:0 24.640,
 'binary_accuracy': tensor cuda:0 0.937}

In [20]:
evaluate_performance_continuous(2)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


  0%|          | 0/571 [00:00<?, ?it/s]/tmp/ipykernel_691660/4081346870.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(file_path)
100%|██████████| 

{'total_loss': 317.11458006646944,
 'binary_accuracy': tensor cuda:0 0.940,
 'mse_loss': tensor[1] cuda:0 [317.111]}

In [21]:
cfg = load_config()

In [22]:
cfg.dataset.test_subject_indices

[1, 2, 13, 24, 26, 27, 29, 34, 35, 41, 42, 43, 45, 46, 47, 48, 52, 55, 56, 57, 60, 62, 67, 69, 72, 73, 79, 80, 86, 88, 92, 102]

In [24]:
results = {subj_index: {} for subj_index in cfg.dataset.test_subject_indices}
for subj_index in cfg.dataset.test_subject_indices:
    results[subj_index]["continuous"] = evaluate_performance_continuous(subj_index)
    final_model = load_model(cfg, start_index=start_indices[subj_index][0], subject_index=subj_index)
    results[subj_index]["final"] = evaluate_performance_final_pretrain(final_model, subj_index)
    results[subj_index]["pretrain"] = evaluate_performance_final_pretrain(pretrained_model, subj_index)




Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


  0%|          | 0/360 [00:00<?, ?it/s]/tmp/ipykernel_691660/4081346870.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(file_path)
100%|██████████| 

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif ...


...

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif ...


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available
Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [25]:
np.save("results.npy", results, allow_pickle=True)